# 🧠 Projet NLP & RAG - Intelligence Artificielle
**Réalisé par : Amina & Firdawss** | **ENSA Al Hoceima - ID2**

Bienvenue dans le Notebook officiel et autonome de notre projet. Ce document contient **l'intégralité du code source** de notre pipeline en deux phases principales :
1. **Phase 1 : Boîte à outils NLP** (Pipelines de base avec HuggingFace)
2. **Phase 2 : Chatbot RAG Intelligent** (Génération Augmentée par la Recherche sur nos propres cours PDF)

Puisque ce Notebook est conçu pour tourner sur Google Colab, tout le code des différentes classes a été regroupé ici dans des cellules exécutables pour faciliter l'évaluation par le professeur.

---

## 🛠️ Étape 0 : Installation des Dépendances et Téléchargement des PDF
Nous installons les bibliothèques nécessaires et téléchargeons les fichiers PDF originaux depuis notre dépôt GitHub.

In [1]:
!pip install -q transformers torch sentence-transformers faiss-cpu PyPDF2
!git clone https://github.com/amina-dourdi/nlp-transformers-rag.git
import os
os.chdir("nlp-transformers-rag")
print("✅ Environnement préparé et PDF téléchargés !")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 13.8 MB/s eta 0:00:00
Cloning into 'nlp-transformers-rag'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 138 (delta 62), reused 105 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 537.03 KiB | 3.07 MiB/s, done.
Resolving deltas: 100% (62/62), done.
✅ Environnement préparé et PDF téléchargés !


---
## ⚙️ Configuration Globale (`config.py`)
Définition des hyperparamètres et des modèles HuggingFace utilisés dans le projet.

In [2]:
# src/config.py
import os

# General and safe models have been selected to run on any device
NLP_SENTIMENT_MODEL = "distilbert-base-uncased-finetuned-sst-2-english"
NLP_QA_MODEL = "deepset/roberta-base-squad2"
NLP_CLASSIFIER_MODEL = "bhadresh-savani/distilbert-base-uncased-emotion"
NLP_SUMMARIZATION_MODEL = "facebook/bart-large-cnn"

# Choix des modèles RAG
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
GENERATIVE_LLM_MODEL = "bigscience/bloomz-560m"

# Hyperparamètres RAG
CHUNK_SIZE = 600       # Size of each chunk (in characters) to preserve context accuracy
CHUNK_OVERLAP = 100    # Overlap between chunks to avoid splitting ideas and sentences

VECTOR_DB_PATH = "data/faiss_index"
CORPUS_DIR = "data/corpus"


---
## 📚 Phase 1 : Boîte à outils NLP (`nlp_pipelines.py`)
Fonctions de classification, résumé et Q&A classiques utilisant les `pipeline`.

In [3]:
# src/core_nlp/nlp_pipelines.py
# ==========================================
# PHASE 1 : exploration et pipelines NLP
# ==========================================

# 1. Analyse de sentiment (sentiment-analysis)
# 2. Question Answering (QA) - extraction de réponse

# 3. Classification de texte (text-classification)
# 4. Résumé automatique (summarization)

from transformers import AutoTokenizer, pipeline, AutoModelForQuestionAnswering, AutoModelForSeq2SeqLM
from src.config import *
import torch


# ============================================================================
# GLOBAL CACHES INITIALIZATION
# ============================================================================
_sentiment_pipeline = None
_qa_tokenizer = None
_qa_model = None
_classification_pipeline = None

# Summarization is loaded explicitly and reliably to prevent Pipeline Registry errors
_summarization_tokenizer = None
_summarization_model = None

def init_pipelines():
    """
    [Task F3 - Corrected] Initialize all Hugging Face models used in the project.
    Fixes the Hugging Face KeyError by explicitly loading the Seq2Seq architecture for BART.
    """
    global _sentiment_pipeline, _qa_tokenizer, _qa_model
    global _classification_pipeline, _summarization_tokenizer, _summarization_model

    print("==================================================")
    print("🔄 INITIALIZATION OF ALL PROJECT MODEL CACHES...")
    print("==================================================")

    # ────────────────────────────────────
    print("🔄 Initializing Sentiment Analysis Pipeline...")
    _sentiment_pipeline = pipeline("sentiment-analysis", model=NLP_SENTIMENT_MODEL)

    # ────────────────────────────────────
    print("🔄 Initializing Question Answering Components (Manual Setup)...")
    _qa_tokenizer = AutoTokenizer.from_pretrained(NLP_QA_MODEL)
    _qa_model = AutoModelForQuestionAnswering.from_pretrained(NLP_QA_MODEL)

    # ────────────────────────────────────
    print("🔄 Initializing Text Classification Pipeline...")
    _classification_pipeline = pipeline("text-classification", model=NLP_CLASSIFIER_MODEL)

    # ────────────────────────────────────
    print("🔄 Initializing Explicit Text Summarization Components...")
    # Explicitly load the encoder-decoder architecture specialized for summarization
    # to avoid issues in recent versions of the transformers library
    _summarization_tokenizer = AutoTokenizer.from_pretrained(NLP_SUMMARIZATION_MODEL)
    _summarization_model = AutoModelForSeq2SeqLM.from_pretrained(NLP_SUMMARIZATION_MODEL)

    print("\n✅ All pipelines are successfully cached and ready without errors!")

# ============================================================================


def analyze_sentiment(text: str) -> dict:
    global _sentiment_pipeline
    if _sentiment_pipeline is None:
        init_pipelines()
    result = _sentiment_pipeline(text)
    return result[0]

# ============================================================================

def answer_question(question: str, context: str) -> dict:
    global _qa_tokenizer, _qa_model
    if _qa_tokenizer is None or _qa_model is None:
        init_pipelines()

    inputs = _qa_tokenizer(question, context, return_tensors="pt")
    with torch.no_grad():
        outputs = _qa_model(**inputs)

    start_idx = torch.argmax(outputs.start_logits)
    end_idx = torch.argmax(outputs.end_logits) + 1

    answer_tokens = inputs["input_ids"][0][start_idx:end_idx]
    answer = _qa_tokenizer.decode(answer_tokens, skip_special_tokens=True)

    score = (torch.max(outputs.start_logits) + torch.max(outputs.end_logits)).item() / 2.0
    return {"answer": answer, "score": score}

# ============================================================================

def classify_text(text: str) -> dict:
    global _classification_pipeline
    if _classification_pipeline is None:
        init_pipelines()
    try:
        prediction = _classification_pipeline(text)[0]
        return {
            "status": "success",
            "label": prediction["label"],
            "confidence": round(prediction["score"], 4)
        }
    except Exception as e:
        return {"status": "error", "message": str(e)}

# ============================================================================

def summarize_text(text: str, max_length: int = 130, min_length: int = 30) -> dict:
    """Generates a summary using an explicit Seq2Seq generation loop to guarantee execution."""
    global _summarization_tokenizer, _summarization_model
    if _summarization_tokenizer is None or _summarization_model is None:
        init_pipelines()
    try:
        # Convert the input text into token IDs for the BART model
        inputs = _summarization_tokenizer(text, max_length=1024, truncation=True, return_tensors="pt")

        # Generation loop following the recommended settings from the book
        summary_ids = _summarization_model.generate(
            inputs["input_ids"],
            num_beams=4,
            max_length=max_length,
            min_length=min_length,
            length_penalty=2.0,
            early_stopping=True
        )

        # Decode and return the final generated summary
        summary_text = _summarization_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        return {"summary_text": summary_text}
    except Exception as e:
        return {"summary_text": f"Error during summarization: {str(e)}"}

---
## 🤖 Phase 2 : Ingestion des Documents RAG (`document_loader.py`)
Lecture des PDF avec `PyPDF2` et découpage intelligent en 'Chunks' pour respecter la fenêtre de contexte.

In [6]:
! pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 3.9 MB/s eta 0:00:00


In [7]:
# src/rag_engine/document_loader.py
# ============================================================================
# PHASE 2: DESIGN ARCHITECTURE: DOCUMENT LOADING & GRANULAR CHUNKING
# ============================================================================
# 1. The Problem (Why We Cannot Ingest Full Documents):
# When a user queries the RAG system, the answer often resides within a specific
# paragraph inside a large corpus (e.g., a 14-page lecture or a 500-page book).
# Attempting to send raw, unsegmented documents directly to a Large Language Model (LLM)
# introduces three critical software engineering vulnerabilities:
#
# - Context Window Saturation: LLMs operate under rigid computational constraints
#   known as token limits. Injecting full-length academic publications overflows
#   this boundary, crashing the runtime via memory exhaustion or context overflow.
# - "Lost in the Middle" Phenomenon: Deep learning research proves that LLMs suffer
#   from severe attention degradation when processing heavy prompts; they retain
#   information at the absolute margins (beginning and end) while ignoring facts
#   buried in the middle.
# - Computation & Latency Bottlenecks: Processing thousands of irrelevant words
#   for a simple query spikes latency, creating a sluggish user experience.
#
# 2. The Solution (Homogeneous Text Segmentation & Overlap Retention):
# To establish an efficient search space, we transform monolithic files into dense,
# manageable data units through a two-fold engineering strategy implemented below:
#
# A. Strict Size Regularization (CHUNK_SIZE = 600):
# Text extraction standardizes document structures page-by-page. We slice raw strings
# into uniform packets of 600 characters. This dimension isolates specific definitions,
# concepts, or equations, ensuring that our FAISS vector index retrieves exactly
# the relevant context needed, rather than drowning the generator in noise.
#
# B. Sliding Window Semantic Preservation (CHUNK_OVERLAP = 100):
# Arbitrary hard cuts threaten text integrity, often splitting key phrases across
# separate blocks and breaking their semantic meaning. By implementing a 100-character
# "Overlap" (chevauchement), trailing characters of a chunk are repeated at the onset
# of the next. This sliding window guarantees absolute semantic continuity and contextual
# preservation across boundary layers.
# ============================================================================

from typing import List, Dict
import os
from src.config import CHUNK_SIZE, CHUNK_OVERLAP
from pypdf import PdfReader # pip install pypdf



def load_pdf_documents(folder_path: str) -> List[Dict[str, str]]:
    """
    Scans a folder and extracts text from every page of every PDF file.
    Each page becomes a dictionary containing its text and exact source.
    """
    documents = []

    if not os.path.exists(folder_path):
        print(f"⚠️ The folder '{folder_path}' does not exist! Create it or verify the path.")
        return documents

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            filepath = os.path.join(folder_path, filename)
            print(f"📄 Reading: '{filename}'...")

            try:
                reader = PdfReader(filepath)
                for page_num, page in enumerate(reader.pages, start=1):
                    text = page.extract_text()

                    # Keep only pages that contain readable text
                    if text and text.strip():
                        documents.append({
                            "text": text.strip(),
                            "source": f"{filename} - Page {page_num}"
                        })

                print(f"  ✅ Extraction successful: {len(reader.pages)} pages found.")

            except Exception as e:
                print(f"  ❌ Error while reading '{filename}': {e}")

    print(f"\n📚 Total: {len(documents)} pages extracted from the corpus.")
    return documents


def get_chunks(documents: List[Dict[str, str]]) -> List[Dict[str, str]]:
    """
    Splits document pages into homogeneous chunks
    using a sliding window with overlap.
    """
    chunks = []

    for doc in documents:
        text = doc["text"]
        source = doc["source"]

        # If the text is smaller than the target size, keep it as a single chunk
        if len(text) <= CHUNK_SIZE:
            chunks.append({
                "text": text,
                "source": source
            })
            continue

        # Sliding Window chunking algorithm
        start = 0
        chunk_num = 1

        while start < len(text):
            end = start + CHUNK_SIZE

            # Ne pas couper les mots en plein milieu : trouver le dernier espace
            if end < len(text):
                last_space = text.rfind(' ', start, end)
                if last_space != -1 and last_space > start:
                    end = last_space

            chunk_text = text[start:end]

            if chunk_text.strip():
                chunks.append({
                    "text": chunk_text.strip(),
                    "source": f"{source} (Chunk {chunk_num})"
                })
                chunk_num += 1

            # Move forward while preserving overlap for semantic continuity
            start = end - CHUNK_OVERLAP

            # Pour l'overlap, s'assurer de commencer au début d'un mot
            if start > 0 and start < len(text):
                next_space = text.find(' ', start, end)
                if next_space != -1:
                    start = next_space + 1

    print(
        f"✂️ Chunking completed: {len(chunks)} chunks created from {len(documents)} pages."
    )
    return chunks

## 📊 Base de Données Vectorielle (`vector_store.py`)
Encodage des Chunks avec `all-MiniLM-L6-v2` et indexation mathématique avec **FAISS**.

In [8]:
# src/rag_engine/vector_store.py

# ============================================================================
# PHASE 2 : DESIGN ARCHITECTURE: INDEX PERSISTENCE & OPTIMIZATION
# ============================================================================
# 1. The Problem (Without Vector Persistence):
# During the document processing phase, text sequences are divided into discrete chunks.
# To make these fragments searchable by the RAG architecture, they must pass through
# an Embedding Model. This model performs a computationally heavy mathematical process,
# transforming natural language strings into high-dimensional numerical dense arrays (Vectors).
#
# Without a dedicated persistence layer, a severe "Performance Bottleneck" arises:
# - Every time the Streamlit application instantiates or reboots, the CPU is forced
#   to re-compute the entire vector space for all chunks from scratch.
# - While fast for miniature setups, scaled knowledge bases (e.g., a 500-page book)
#   would trigger minutes of computational latency at every runtime init, rendering
#   the production pipeline highly inefficient.
#
# 2. The Solution (With Index Serialization & Metadata Mapping):
# The core architectural goal is "Compute Once, Use Indefinitely". We bypass redundant
# transformations by storing the finalized states directly into the file system as
# persistent binary and structural configurations.
# This is orchestrated via two complementary methods:
#
# A. save_index():
# Meta's FAISS library provides ultra-fast nearest-neighbor lookup matrices, but it is
# inherently "blind" to original text strings (it strictly indexes index keys and numerical shapes).
# To circumvent this, we implement a hybrid storage mechanism:
#   - Serializing the pure numerical coordinate matrices into a specialized binary file (faiss.index).
#   - Synchronously writing a descriptive text catalog (source files, page numbers, text content)
#     into a parallel metadata file formatted in structured JSON (faiss.index.docs.json).
#
# B. load_index():
# When the app reboots, rather than calling the transformer model to run redundant forward passes,
# the system reaches straight to the storage directories. It reloads both the mathematical index
# and text dictionaries back into active memory in milliseconds, guaranteeing lightning-fast,
# zero-latency execution.
# ============================================================================


from sentence_transformers import SentenceTransformer
from typing import List, Dict
from src.config import *
import numpy as np
import faiss
import json



class VectorStoreManager:
    def __init__(self, model_name: str = EMBEDDING_MODEL_NAME, index_path: str = VECTOR_DB_PATH):
        """
        Manages semantic vector index generation, storage, and persistence.
        """
        self.model_name = model_name
        self.index_path = index_path
        self.documents = []  # To cache original string contexts and sources (Metadata)
        self.index = None    # Core FAISS index tracker

        print(f"🔄 Loading sentence embedding model: '{model_name}'...")
        self.embedding_model = SentenceTransformer(model_name)
        print("✅ Semantic embedding transformer ready!")

    def build_and_save_index(self, chunks: List[Dict[str, str]]):
        """
        Computes textual embeddings, instantiates an L2 flat FAISS vector database,
        and automatically persists everything to disk.
        """
        self.documents = chunks
        raw_texts = [c["text"] for c in chunks]

        print(f"🔄 Computing semantic vectors for {len(raw_texts)} chunks...")
        embeddings = self.embedding_model.encode(raw_texts, show_progress_bar=True)
        embeddings = np.array(embeddings).astype("float32")

        # Build FAISS index mapped to the specific structural dimension of the model
        vector_dimension = embeddings.shape[1]
        print(f"📐 Vector dimensional space: {vector_dimension}")

        # Utilizing Euclidean (L2) Distance for deep semantic retrieval match
        self.index = faiss.IndexFlatL2(vector_dimension)
        self.index.add(embeddings)
        print(f"✅ FAISS index established with {self.index.ntotal} registered vectors.")

        # Automatically persist components to local memory fields (Task F5)
        self.save_index()

    def save_index(self):
        """
        [Task F5 Execution] Serializes the binary FAISS index and structural JSON metadata to disk.
        """
        if self.index is None:
            print("⚠️ No structural index initialized to preserve!")
            return

        # Guarantee parent directories exist prior to serialization
        os.makedirs(os.path.dirname(self.index_path), exist_ok=True)

        # A. Persist the mathematical vector points inside the binary index file
        faiss.write_index(self.index, self.index_path)
        print(f"💾 Binary FAISS vectors successfully written to: '{self.index_path}'")

        # B. Persist raw textual strings since FAISS strictly manages vectors alone
        metadata_json_path = self.index_path + ".docs.json"
        with open(metadata_json_path, "w", encoding="utf-8") as f:
            json.dump(self.documents, f, ensure_ascii=False, indent=2)
        print(f"💾 Accompanying JSON text mappings written to: '{metadata_json_path}'")

    def load_index(self) -> bool:
        """
        [Task F5 Execution] Reloads cached FAISS databases and document dictionary catalogs from disk.
        Returns True if operational, False if index files are missing.
        """
        metadata_json_path = self.index_path + ".docs.json"

        if not os.path.exists(self.index_path) or not os.path.exists(metadata_json_path):
            print(f"⚠️ Pre-computed database cache not found at target: '{self.index_path}'")
            return False

        # A. Reload vector spaces directly from local binarized files
        self.index = faiss.read_index(self.index_path)

        # B. Reload matching metadata sequences synchronously
        with open(metadata_json_path, "r", encoding="utf-8") as f:
            self.documents = json.load(f)

        print(f"📂 [Success] Locally persisted FAISS index reloaded ({self.index.ntotal} vectors).")
        print(f"📂 Synchronized {len(self.documents)} context sources from disk metadata cache.")
        return True

    def search_top_k(self, query: str, k: int = 3) -> List[Dict[str, str]]:
        """
        Transforms incoming natural language queries into mathematical arrays,
        queries the FAISS matrix, and returns the top k semantic matches.
        """
        if self.index is None:
            print("⚠️ Vector storage index not operational! Initialize or run data ingestion first.")
            return []

        # Encode user string query into the exact same vector dimension
        query_vector = self.embedding_model.encode([query])
        query_vector = np.array(query_vector).astype("float32")

        # Execute mathematical flat spatial L2 nearest-neighbors search
        distances, target_indices = self.index.search(query_vector, k)

        retrieved_contexts = []
        for index_point in target_indices[0]:
            if index_point != -1 and index_point < len(self.documents):
                retrieved_contexts.append(self.documents[index_point])

        return retrieved_contexts


## 🧠 Générateur RAG Augmenté (`llm_generator.py`)
Modèle `Bloomz-560m` qui génère la réponse finale en fusionnant la question avec les documents extraits par FAISS.

In [9]:
# src/generator/llm_generator.py
# ==========================================
# PHASE 2 : RAG - Générateur LLM
# ==========================================

#  API HuggingFace Inference
# Pour obtenir un token gratuit :
#   1. Créer un compte sur https://huggingface.co
#   2. Settings → Access Tokens → New Token
#   3. Définir la variable d'environnement HF_TOKEN
#      ou remplacer directement dans le code ci-dessous.

import os
from typing import List, Dict
from huggingface_hub import InferenceClient
from src.config import GENERATIVE_LLM_MODEL


class RAGGenerator:
    def __init__(self, model_name: str = GENERATIVE_LLM_MODEL, token: str = None):
        """
        Initialise le client d'inférence HuggingFace.

        Args:
            model_name (str): Nom du modèle sur HuggingFace Hub.
            token (str): Token d'API HuggingFace. Si None, lit la variable
                         d'environnement HF_TOKEN.
        """
        # Récupérer le token : paramètre > variable d'environnement
        self.token = token or os.environ.get("HF_TOKEN", "")

        if not self.token:
            print(
                "⚠️  Aucun token HuggingFace détecté !\n"
                "    → Définissez la variable d'environnement HF_TOKEN\n"
                "    → Ou passez le token en paramètre : RAGGenerator(token='hf_...')"
            )

        self.model_name = model_name
        self.client = InferenceClient(
            model=self.model_name,
            token=self.token,
        )
        print(f"✅ Générateur LLM connecté au modèle '{self.model_name}'")

    # ──────────────────────────────────────────────
    # Construction du Prompt Augmenté (RAG)
    # ──────────────────────────────────────────────

    def build_prompt(self, query: str, contexts: List[Dict[str, str]]) -> str:
        """
        Formate le prompt augmenté avec les contextes trouvés par Firdawss.

        Le prompt est structuré pour :
        1. Donner un rôle clair au LLM (assistant pédagogique).
        2. Lui fournir le contexte extrait des documents (chunks FAISS).
        3. Lui interdire d'inventer des informations (garde-fou anti-hallucination).
        """
        combined_context = "\n\n".join([
            f"[Source: {c['source']}]\n{c['text']}"
            for c in contexts
        ])

        prompt = f"""Vous êtes un professeur. Répondez à la question en utilisant le contexte.

Contexte :
{combined_context}

---
Exemple :
Question : Qu'est-ce que le NLP ?
Réponse : Le NLP (Natural Language Processing) est une branche de l'IA qui permet aux ordinateurs de comprendre le langage humain.
---

Question : {query}
Réponse :"""
        return prompt

    # ──────────────────────────────────────────────
    # Génération AVEC RAG (contexte fourni)
    # ──────────────────────────────────────────────

    def generate(self, query: str, contexts: List[Dict[str, str]]) -> str:
        """
        Assemble le prompt augmenté et génère la réponse via l'API HuggingFace.

        Args:
            query (str): La question de l'utilisateur.
            contexts (List[Dict]): Les chunks pertinents trouvés par FAISS.
                                   Chaque dict contient 'text' et 'source'.

        Returns:
            str: La réponse générée par le LLM.
        """
        prompt = self.build_prompt(query, contexts)

        try:
            response = self.client.text_generation(
                prompt,
                max_new_tokens=100,
                do_sample=False
            )
            if not response or not response.strip():
                raise ValueError("Réponse API vide")
            return response.strip()

        except Exception as e:
            # 🚀 FALLBACK LOCAL : Exécution locale d'un petit LLM parlant français si l'API crashe
            try:
                from transformers import pipeline
                if not hasattr(self, "local_pipeline"):
                    try:
                        import streamlit as st
                        with st.spinner("L'API a échoué. Téléchargement d'un modèle local (Bloomz)..."):
                            self.local_pipeline = pipeline("text-generation", model=GENERATIVE_LLM_MODEL, device_map="auto")
                    except ImportError:
                        print("L'API a échoué. Téléchargement d'un modèle local (Bloomz)...")
                        self.local_pipeline = pipeline("text-generation", model=GENERATIVE_LLM_MODEL, device_map="auto")

                res = self.local_pipeline(prompt, max_new_tokens=100, do_sample=True, temperature=0.7, return_full_text=False)
                return res[0]["generated_text"].strip()
            except Exception as e_local:
                return f"❌ L'API a échoué ({type(e).__name__}) ET le mode hors-ligne a échoué ({e_local})"

    # ──────────────────────────────────────────────
    # Génération SANS RAG (pour la comparaison)
    # ──────────────────────────────────────────────

    def generate_without_rag(self, query: str) -> str:
        """
        Génère une réponse SANS contexte RAG.

        Utilisé pour la démonstration comparative :
        on montre que sans les documents, le LLM peut halluciner
        ou donner des réponses vagues / incorrectes.

        Args:
            query (str): La question de l'utilisateur.

        Returns:
            str: La réponse générée sans contexte.
        """
        prompt = f"""Répondez à la question suivante de manière claire et détaillée :

Question : {query}

Réponse :"""

        try:
            response = self.client.text_generation(
                prompt,
                max_new_tokens=100,
                do_sample=False
            )
            if not response or not response.strip():
                raise ValueError("Réponse API vide")
            return response.strip()

        except Exception as e:
            try:
                from transformers import pipeline
                if not hasattr(self, "local_pipeline"):
                    try:
                        import streamlit as st
                        with st.spinner("L'API a échoué. Téléchargement d'un modèle local (Bloomz)..."):
                            self.local_pipeline = pipeline("text-generation", model=GENERATIVE_LLM_MODEL, device_map="auto")
                    except ImportError:
                        print("L'API a échoué. Téléchargement d'un modèle local (Bloomz)...")
                        self.local_pipeline = pipeline("text-generation", model=GENERATIVE_LLM_MODEL, device_map="auto")

                res = self.local_pipeline(prompt, max_new_tokens=100, return_full_text=False)
                return res[0]["generated_text"].strip()
            except Exception as e_local:
                return f"❌ L'API a échoué ({type(e).__name__}) ET le mode hors-ligne a échoué ({e_local})"


---
# 🚀 Éxécution & Démonstration Complète
Test du pipeline RAG avec une vraie question pour démontrer les capacités de l'Intelligence Artificielle sourcée.

In [10]:
! pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 49.0 MB/s eta 0:00:00


In [11]:
print("=== 1. INDEXATION DES PDF DE COURS ===")
docs = load_pdf_documents("data/corpus")
chunks_list = get_chunks(docs)
manager = VectorStoreManager(EMBEDDING_MODEL_NAME, VECTOR_DB_PATH)
manager.build_and_save_index(chunks_list)
print("\n=== 2. QUESTION UTILISATEUR ===")
q = "Qu'est-ce que le mécanisme d'attention dans les Transformers ?"
print(f"Question posée : '{q}'")
res = manager.search_top_k(q, k=3)
print("\n=== 3. GÉNÉRATION LLM (SANS RAG) ===")
gen = RAGGenerator()
print("❌ ", gen.generate_without_rag(q))
print("\n=== 4. GÉNÉRATION LLM (AVEC RAG) ===")
print("✅ ", gen.generate(q, res))
print("\n=== 5. PREUVE DES SOURCES ===")
for i, r in enumerate(res, 1):
    print(f"Source {i} : {r['source']}")

=== 1. INDEXATION DES PDF DE COURS ===
📄 Reading: '02_Les_Transformers.pdf'...
  ✅ Extraction successful: 1 pages found.
📄 Reading: '03_Systemes_RAG.pdf'...
  ✅ Extraction successful: 1 pages found.
📄 Reading: '01_Introduction_NLP.pdf'...
  ✅ Extraction successful: 1 pages found.

📚 Total: 3 pages extracted from the corpus.
✂️ Chunking completed: 14 chunks created from 3 pages.
🔄 Loading sentence embedding model: 'all-MiniLM-L6-v2'...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Passing `generation_config` together w

❌  système de montage de films

=== 4. GÉNÉRATION LLM (AVEC RAG) ===
✅  Self-Attention

=== 5. PREUVE DES SOURCES ===
Source 1 : 02_Les_Transformers.pdf - Page 1 (Chunk 2)
Source 2 : 02_Les_Transformers.pdf - Page 1 (Chunk 1)
Source 3 : 02_Les_Transformers.pdf - Page 1 (Chunk 4)
